# Notebook 05 — Inference, Alerts & Forecast Visualization
## Pearls AQI Predictor · Hyderabad, Pakistan

**Objective:** Load the best trained model, run inference on the latest data, generate the 72-hour
forecast, check alert thresholds, and visualize everything the way the dashboard would.

**What we do:**
1. Load best model + latest feature data
2. Run `InferenceEngine` to generate 24h/48h/72h predictions
3. Classify each prediction (Good → Hazardous)
4. Check alert thresholds (AQI ≥ 200)
5. Visualize the full forecast timeline
6. Compare Open-Meteo forecast AQI vs our model's prediction
7. Generate a summary report ready for dashboard display

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
from datetime import datetime, timedelta

from feature_store.feature_builder import FeatureBuilder
from models.inference import InferenceEngine
from models.registry import get_latest_model
from utils.config import get
from utils.storage import load_parquet, save_json
from utils.aqi_utils import (
    classify_aqi, is_alert_level, category_color, category_advice, AQICategory
)
from utils.time_utils import now_local, format_iso

plt.style.use('dark_background')
sns.set_palette('viridis')
%matplotlib inline

pd.set_option('display.max_columns', 60)

# AQI category colors for plotting
CAT_COLORS = {
    'Good': '#00e400',
    'Moderate': '#ffff00',
    'Unhealthy for Sensitive Groups': '#ff7e00',
    'Unhealthy': '#ff0000',
    'Very Unhealthy': '#8f3f97',
    'Hazardous': '#7e0023',
}

In [ ]:
# Load data and build features
DATA_DIR = Path(get('storage.data_dir', '../data'))
merged_path = DATA_DIR / 'processed' / 'merged_hourly' / 'merged_latest.parquet'

try:
    df = load_parquet(merged_path)
    builder = FeatureBuilder(df)
    featured = builder.build_all()
    print(f'✅ Loaded {len(featured)} featured rows from real data')
except FileNotFoundError:
    print('⚠️ No merged data. Creating synthetic forecast demo...')
    np.random.seed(42)
    n = 200
    base = datetime.now().replace(minute=0, second=0, microsecond=0)
    ts = [base - timedelta(hours=i) for i in range(n, 0, -1)]
    trend = np.cumsum(np.random.randn(n) * 3)
    season = np.sin(np.arange(n) * 2 * np.pi / 24) * 12
    aqi_values = np.clip(80 + trend + season, 0, 350)
    
    df = pd.DataFrame({
        'timestamp': ts,
        'aqi': aqi_values,
        'pm2_5': aqi_values * 0.8 + np.random.randn(n) * 5,
        'pm10': aqi_values * 1.1 + np.random.randn(n) * 10,
        'temperature_2m': 28 + np.sin(np.arange(n)*2*np.pi/24)*6 + np.random.randn(n)*2,
        'relative_humidity_2m': np.clip(55 + np.random.randn(n)*12, 10, 95),
        'wind_speed_10m': np.abs(4 + np.random.randn(n)*2),
        'wind_direction_10m': np.random.uniform(0, 360, n),
        'precipitation': np.maximum(0, np.random.randn(n))*2,
        'cloud_cover': np.clip(np.random.uniform(0, 100, n), 0, 100),
        'dew_point_2m': 18 + np.random.randn(n)*3,
        'pressure_msl': 1008 + np.random.randn(n)*3,
    })
    builder = FeatureBuilder(df)
    featured = builder.build_all()
    print(f'   Created {len(featured)} synthetic rows for demo')

print(f'Timestamp range: {featured["timestamp"].min()} → {featured["timestamp"].max()}')

---
## 1. Run Inference Engine

The `InferenceEngine` loads the best model from MLflow (or local fallback) and runs predictions.

In [ ]:
engine = InferenceEngine()
forecast = engine.predict(featured)

print(f'✅ Inference complete')
print(f'   Current AQI: {forecast.get("current_aqi", "N/A"):.0f}')
print(f'   Model: {forecast.get("model_info", {}).get("type", "unknown")}')
print(f'   Timestamp: {forecast.get("timestamp")}')
print()

for h in ['24h', '48h', '72h']:
    f = forecast['forecast'].get(h, {})
    print(f'   +{h}: AQI={f.get("aqi", "?")}, Category={f.get("category", "?")}, Alert={"⚠ YES" if f.get("alert") else "✓ No"}')

---
## 2. AQI Category Classification

Map each predicted AQI value to its health category and show the breakdown.

In [ ]:
print('=== AQI Category Breakdown ===\n')

def print_category_row(label, aqi_value):
    cat = classify_aqi(aqi_value)
    alert = '⚠ ALERT!' if is_alert_level(aqi_value) else '✓'
    color = category_color(cat)
    advice = category_advice(cat)
    print(f'{label:<15} | AQI: {aqi_value:6.1f} | {cat.value:<35} | {alert}')
    return cat

current_cat = print_category_row('Current', forecast.get('current_aqi', 0))
print('-' * 90)

for h in ['24h', '48h', '72h']:
    f = forecast['forecast'].get(h, {})
    print_category_row(f'Forecast +{h}', f.get('aqi', 0))

print()
print(f'Health advice: {category_advice(current_cat)}')

# All categories with thresholds
print('\n--- Reference: All AQI Categories ---')
for cat in AQICategory:
    if cat == AQICategory.UNKNOWN:
        continue
    thresholds = get(f'aqi_thresholds.{cat.value.lower().replace(" ", "_")}', [])
    color = category_color(cat)
    print(f'  {cat.value:<35} color={color}')

---
## 3. Alert Check — Hazardous AQI Detection

The system triggers alerts when **AQI ≥ 200** (Very Unhealthy or Hazardous).

In [ ]:
alerts = []

# Check current
current_aqi = forecast.get('current_aqi', 0)
if is_alert_level(current_aqi):
    level = 'Hazardous' if current_aqi >= 300 else 'Very Unhealthy'
    alerts.append({
        'type': 'current',
        'aqi': current_aqi,
        'level': level,
        'message': f'Current AQI is {current_aqi:.0f} — {level}. Limit outdoor activity.'
    })

# Check forecasts
for h in ['24h', '48h', '72h']:
    f = forecast['forecast'].get(h, {})
    if f.get('alert'):
        alerts.append({
            'type': f'forecast_{h}',
            'aqi': f.get('aqi'),
            'level': f.get('category'),
            'message': f'Forecast AQI in {h} is {f.get("aqi", "?")} — {f.get("category")}. Take precautions.'
        })

print(f'=== Alert Summary ===')
print(f'Alert threshold: AQI ≥ {get("alerts.alert_threshold", 200)}')
print(f'Alerts triggered: {len(alerts)}')
print()

if alerts:
    for a in alerts:
        print(f'  ⚠ {a["type"]}: {a["message"]}')
else:
    print('  ✅ No alerts — air quality is within acceptable levels.')

---
## 4. Forecast Visualization — Full Timeline

In [ ]:
# Build a complete forecast timeline
current = forecast.get('current_aqi', 0)

# Past 48 hours + forecast points
aqi_col = 'aqi' if 'aqi' in featured.columns else 'om_forecast_aqi'
past_data = featured[[aqi_col, 'timestamp']].dropna().tail(48) if aqi_col in featured.columns else pd.DataFrame()

fig, ax = plt.subplots(figsize=(18, 7))

# Past AQI (solid line)
if not past_data.empty:
    ax.plot(past_data['timestamp'], past_data[aqi_col], 
            color='#4da6ff', linewidth=1.5, label='Observed AQI', alpha=0.8)
    last_ts = past_data['timestamp'].max()
else:
    last_ts = pd.Timestamp.now()

# Forecast points
forecast_times = [last_ts + timedelta(hours=h) for h in [24, 48, 72]]
forecast_values = [forecast['forecast'][f'{h}h']['aqi'] for h in [24, 48, 72]]
forecast_cats = [forecast['forecast'][f'{h}h']['category'] for h in [24, 48, 72]]

# Draw forecast as markers + dashed line
all_times = [last_ts] + forecast_times
all_values = [current] + forecast_values
ax.plot(all_times, all_values, 'o--', color='#ff7e00', linewidth=2.5, markersize=10, 
        label='Model Forecast', zorder=5)

# Color each forecast point by category
for t, v, cat in zip(forecast_times, forecast_values, forecast_cats):
    color = CAT_COLORS.get(cat, '#808080')
    ax.plot(t, v, 'o', color=color, markersize=14, markeredgewidth=2, 
            markeredgecolor='white', zorder=6)
    ax.annotate(f'{cat}\n({v:.0f})', (t, v), textcoords='offset points',
                xytext=(0, 18), ha='center', fontsize=9, fontweight='bold', color=color)

# Current marker
ax.plot(last_ts, current, 'D', color='#00d4ff', markersize=12, markeredgecolor='white', zorder=6)
ax.annotate(f'Now: {current:.0f}', (last_ts, current), textcoords='offset points',
            xytext=(-10, -20), fontsize=10, fontweight='bold', color='#00d4ff')

# Alert threshold line
alert_threshold = get('alerts.alert_threshold', 200)
ax.axhline(y=alert_threshold, color='#ff3333', linestyle='--', linewidth=1.5, alpha=0.7,
           label=f'Alert threshold ({alert_threshold})')
ax.fill_between([all_times[0], all_times[-1]], alert_threshold, 400, 
                color='#ff3333', alpha=0.05)

# Category background bands
ax.axhspan(0, 50, color='#00e400', alpha=0.03)
ax.axhspan(201, 400, color='#ff3333', alpha=0.05)

ax.set_xlabel('Time (Asia/Karachi)', fontsize=12)
ax.set_ylabel('AQI', fontsize=12)
ax.set_title(f'Hyderabad AQI Forecast — {current:.0f} now → {forecast_values[-1]:.0f} in 72 hours', fontsize=15)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.15)
ax.set_ylim(0, max(max(all_values) * 1.3, alert_threshold + 50))

plt.tight_layout()
plt.show()

---
## 5. Multi-Pollutant Forecast Breakdown

In [ ]:
# Show pollutant trends if available
pollutant_cols = ['pm2_5', 'pm10']
available_pollutants = [c for c in pollutant_cols if c in featured.columns]

if available_pollutants:
    sample = featured[['timestamp'] + available_pollutants].dropna().tail(72)

    fig, axes = plt.subplots(len(available_pollutants), 1, figsize=(16, 4 * len(available_pollutants)), sharex=True)
    if len(available_pollutants) == 1:
        axes = [axes]

    colors = {'pm2_5': '#ff7e00', 'pm10': '#ff3333'}
    for ax, col in zip(axes, available_pollutants):
        ax.fill_between(sample['timestamp'], sample[col], alpha=0.3, color=colors.get(col, '#4da6ff'))
        ax.plot(sample['timestamp'], sample[col], color=colors.get(col, '#4da6ff'), linewidth=2)
        ax.set_ylabel(f'{col.upper()} (μg/m³)')
        ax.set_title(f'{col.upper()} Trend — Last 72 Hours')
        ax.grid(True, alpha=0.2)

    axes[-1].set_xlabel('Time (Asia/Karachi)')
    plt.tight_layout()
    plt.show()

---
## 6. Model vs Open-Meteo Raw Forecast Comparison

Compare our trained model's prediction against the raw Open-Meteo AQI forecast.
This shows the value of training a custom model — it learns the local relationship between weather and AQI.

In [ ]:
om_col = 'om_forecast_aqi'
if om_col in featured.columns:
    om_sample = featured[['timestamp', om_col]].dropna().tail(48)

    if not om_sample.empty:
        fig, ax = plt.subplots(figsize=(16, 6))

        ax.plot(om_sample['timestamp'], om_sample[om_col], 
                color='#9898b8', linewidth=1.5, linestyle=':', alpha=0.6,
                label='Open-Meteo Raw Forecast AQI')

        # Our model forecast
        ax.plot(all_times, all_values, 'o-', color='#ff7e00', linewidth=2.5, markersize=8,
                label='Trained Model Forecast')

        # Observed AQI if available
        if 'aqi' in featured.columns:
            aqi_sample = featured[['timestamp', 'aqi']].dropna().tail(48)
            if not aqi_sample.empty:
                ax.plot(aqi_sample['timestamp'], aqi_sample['aqi'], 
                        color='#00e400', linewidth=1.5, alpha=0.5,
                        label='AQICN Observed AQI (ground truth)')

        ax.set_title('Comparison: Raw Open-Meteo Forecast vs Trained Model vs Observed AQI', fontsize=14)
        ax.set_xlabel('Time (Asia/Karachi)')
        ax.set_ylabel('AQI')
        ax.legend()
        ax.grid(True, alpha=0.2)
        plt.tight_layout()
        plt.show()

        print('💡 The trained model learns the relationship between weather conditions and actual air quality.')
        print('   Open-Meteo provides a generic forecast; our model customizes it for Hyderabad using AQICN labels.')
else:
    print('Open-Meteo forecast AQI column not available for comparison.')

---
## 7. Generate Summary Report

Produce a clean JSON report suitable for the dashboard.

In [ ]:
report = {
    'city': 'Hyderabad, Pakistan',
    'station': 'A546205',
    'generated_at': format_iso(now_local()),
    'current': {
        'aqi': round(current, 1),
        'category': classify_aqi(current).value,
        'category_color': category_color(classify_aqi(current)),
        'health_advice': category_advice(classify_aqi(current)),
        'alert': is_alert_level(current),
    },
    'forecast': {},
    'alerts': alerts,
    'model_info': forecast.get('model_info', {}),
}

for h in ['24h', '48h', '72h']:
    f = forecast['forecast'].get(h, {})
    report['forecast'][f'+{h}'] = {
        'aqi': f.get('aqi'),
        'category': f.get('category'),
        'alert': f.get('alert'),
    }

print(json.dumps(report, indent=2, default=str))

In [ ]:
# Save forecast for API/dashboard consumption
forecast_dir = DATA_DIR / 'processed' / 'predictions'
forecast_dir.mkdir(parents=True, exist_ok=True)

save_json(forecast, forecast_dir / 'forecast_latest.json')
print(f'✅ Forecast saved to: {forecast_dir / "forecast_latest.json"}')

save_json(report, forecast_dir / f'forecast_report_{datetime.now().strftime("%Y%m%d_%H%M")}.json')
print(f'✅ Report saved')

---
## 8. Dashboard Preview — What the User Sees

Simulate what the dashboard home page would display.

In [ ]:
cat = classify_aqi(current)
color = category_color(cat)

print('╔══════════════════════════════════════════╗')
print('║        🌫 PEARLS AQI PREDICTOR          ║')
print('║         Hyderabad, Pakistan              ║')
print('╠══════════════════════════════════════════╣')
print(f'║                                          ║')
print(f'║         Current AQI: {current:>6.0f}            ║')
print(f'║         {cat.value:<34} ║')
print(f'║                                          ║')
print(f'║  {category_advice(cat)[:36]:<36} ║')
print(f'║                                          ║')
print(f'╠══════════════════════════════════════════╣')
print(f'║  Forecast:                               ║')

for h in ['24h', '48h', '72h']:
    f = forecast['forecast'].get(h, {})
    alert_icon = '⚠' if f.get('alert') else '✓'
    print(f'║  +{h:<3} | AQI: {f.get("aqi", "?"):>6.1f} | {f.get("category", "?"):<20} {alert_icon}   ║')

print(f'╠══════════════════════════════════════════╣')
print(f'║  Model: {forecast.get("model_info", {}).get("type", "N/A"):<31} ║')
print(f'║  Updated: {now_local().strftime("%Y-%m-%d %H:%M PKT"):<29} ║')
print(f'╚══════════════════════════════════════════╝')

if alerts:
    print(f'\n⚠ {len(alerts)} ALERT(S) ACTIVE')

---
## 9. Historical Forecast Accuracy (Back-Test)

If we have historical data, we can back-test: for each past timestamp, what would the model
have predicted vs what actually happened?

In [ ]:
# Simple back-test: compare persistence forecast vs actual at each horizon
target_cols = ['target_aqi_24h', 'target_aqi_48h', 'target_aqi_72h']
available_targets = [c for c in target_cols if c in featured.columns]

if available_targets:
    aqi_actual = featured['aqi'] if 'aqi' in featured.columns else featured['om_forecast_aqi']

    fig, ax = plt.subplots(figsize=(16, 5))

    # Show last 72 hours of actual AQI
    last_72 = featured.tail(72).copy()
    ax.plot(last_72['timestamp'], aqi_actual.tail(72), 
            color='#00e400', linewidth=1.5, label='Actual AQI')

    # Overlay: when a 24h-ahead forecast was made, what value did it predict?
    for target_col, shift_h, style in [
        ('target_aqi_24h', 24, {'color': '#4da6ff', 'ls': '--', 'label': '24h-ago Forecast'}),
        ('target_aqi_48h', 48, {'color': '#ff7e00', 'ls': '--', 'label': '48h-ago Forecast'}),
    ]:
        if target_col in featured.columns:
            # The target at time t is the actual AQI at t+shift
            # So the forecast made at (t-shift) predicted this value
            forecast_vals = featured[target_col].shift(shift_h).tail(72)
            ax.plot(last_72['timestamp'], forecast_vals, 
                    linewidth=1, alpha=0.5, **style)

    ax.set_title('Back-Test: Forecast vs Actual (Persistence Baseline)', fontsize=14)
    ax.set_xlabel('Time')
    ax.set_ylabel('AQI')
    ax.legend()
    ax.grid(True, alpha=0.15)
    plt.tight_layout()
    plt.show()

    print('💡 This back-test shows how accurate a naive persistence forecast would have been.')
    print('   The trained ML model should significantly outperform this baseline.')

---
## Summary

| Step | Result |
|------|--------|
| **Inference** | Generated 24h/48h/72h AQI predictions from latest data |
| **Classification** | Each prediction mapped to health category (Good → Hazardous) |
| **Alerts** | Checked all predictions against ≥200 threshold |
| **Visualization** | Full forecast timeline with category-colored markers |
| **Comparison** | Model forecast vs Open-Meteo raw vs AQICN observed |
| **Report** | JSON summary report saved for dashboard consumption |
| **Back-test** | Historical accuracy check using persistence baseline |

---

## 🎉 End-to-End Pipeline Complete

| Notebook | What It Covers |
|----------|---------------|
| **01** Data Ingestion | Fetch Open-Meteo + AQICN, normalize, validate, merge, backfill |
| **02** Feature Engineering | 50+ features: time, lag, rolling, weather, interactions, targets |
| **03** Model Training | 7 models, walk-forward validation, RMSE/MAE/R² comparison |
| **04** SHAP Explainability | TreeExplainer, global/local importance, correlation fallback |
| **05** Inference & Alerts | Forecast generation, alert detection, dashboard-ready visualizations |

The system is now ready for deployment: `docker compose up --build`